In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer


NAME = "sft_ckpt"   
EPOCHS, BATCH, LR, MAX_LEN, N_SAMPLES = 1, 4, 1e-5, 512, 2000
device = "cuda" if torch.cuda.is_available() else "cpu"


class RewardModel(nn.Module):
    """Transformer backbone + linear head -> one scalar per sequence."""

    def __init__(self, name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name)
        self.head = nn.Linear(self.backbone.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        last = attention_mask.sum(dim=1) - 1                 # index of last real token (right padding)
        h_last = h[torch.arange(h.size(0), device=h.device), last]
        return self.head(h_last).squeeze(-1)                 # (batch,)


def bt_loss(r_chosen, r_rejected):
    """Bradley-Terry: maximise P(chosen > rejected) = sigmoid(r_c - r_r)."""
    return -F.logsigmoid(r_chosen - r_rejected).mean()


if __name__ == "__main__":
    tok = AutoTokenizer.from_pretrained(NAME)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split=f"train_prefs[:{N_SAMPLES}]")

    def fmt(ex, key):
        return ex["prompt"] + "\n\nAnswer: " + ex[key][-1]["content"] + tok.eos_token

    data = [(fmt(ex, "chosen"), fmt(ex, "rejected")) for ex in ds]
    data = [d for d in data if max(len(tok(d[0]).input_ids), len(tok(d[1]).input_ids)) < MAX_LEN]
    print(f"{len(data)} / {N_SAMPLES} pairs kept")

    def collate(batch):
        chosen, rejected = zip(*batch)
        enc = tok(list(chosen) + list(rejected), return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_LEN, add_special_tokens=False)
        return {k: v.to(device) for k, v in enc.items()}

    rm = RewardModel(NAME).to(device)
    opt = torch.optim.AdamW(rm.parameters(), lr=LR)
    rm.train()
    for epoch in range(EPOCHS):
        for step, batch in enumerate(DataLoader(data, batch_size=BATCH, shuffle=True, collate_fn=collate)):
            r = rm(batch["input_ids"], batch["attention_mask"])
            r_c, r_r = r.chunk(2)                            # first half chosen, second half rejected
            loss = bt_loss(r_c, r_r)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(rm.parameters(), 1.0)
            opt.step(); opt.zero_grad()
            if step % 10 == 0:
                acc = (r_c > r_r).float().mean().item()
                print(f"epoch {epoch} step {step} loss {loss.item():.3f} acc {acc:.2f}")

    torch.save(rm.state_dict(), "rm.pt")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


OSError: sft_ckpt is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`